# Parameter Error Figures

Generate per-parameter error plots for each equation, save each as a vector PDF, then merge them into one stacked vector PDF with gray separators and equation subtitles.

In [ ]:
# 1) Imports and project-root setup
from pathlib import Path
from typing import Dict, List

import os
import sys
import importlib

import matplotlib.pyplot as plt
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("CWD:", Path.cwd())
print("PROJECT_ROOT:", PROJECT_ROOT)

from src.utils.configs.training_config import load_config, build_run_plan
import src.utils.evaluation.evaluation as ev
import src.utils.visualisations.visualise_results as vis

importlib.reload(ev)
importlib.reload(vis)


Build selected runs (best validation combination), create per-parameter error plots, and save each equation figure as vector PDF.

In [ ]:
USE_BEST = True
MODEL_ORDER = ["fno", "cape_fno", "late_fusion"]

TRAJ_PARAM_INDEX = {
    "advection": 0,
    "burgers": 0,
    "reactiondiffusion": 1,
    "reactiondiffusion2d": 0,
}

PARAM_LABEL = {
    "advection": r"$\beta$",
    "burgers": r"$\nu$",
    "reactiondiffusion": r"$\nu$",
    "reactiondiffusion2d": r"$k$",
}

SAVE_NAME = {
    "advection": "parametererror_advection_A.pdf",
    "burgers": "parametererror_burgers_B.pdf",
    "reactiondiffusion": "parametererror_reactiondiffusion_C.pdf",
    "reactiondiffusion2d": "parametererror_reactiondiffusion2d_D.pdf",
}

SUBTITLE = {
    "advection": "A. 1D advection",
    "burgers": "B. 1D burgers",
    "reactiondiffusion": "C. 1D reaction-diffusion",
    "reactiondiffusion2d": "D. 2D reaction-diffusion",
}

def _build_selected_runs(equation: str) -> List[Dict]:
    config_path = f"configs/training/{equation}_benchmark.yaml"
    cfg = load_config(config_path)
    runs = build_run_plan(cfg)

    val_rows: List[Dict] = []
    for run_cfg in runs:
        val_rows.append(ev.evaluate_run_validation(run_cfg, use_best=USE_BEST))

    val_df = pd.DataFrame(val_rows)
    _, best_combo = ev.select_best_combinations(val_df)

    selected = val_df.copy()
    selected["combination"] = selected["run_name"].map(ev.combination_key)
    selected = selected.merge(
        best_combo[["model", "combination"]],
        on=["model", "combination"],
        how="inner",
    )

    run_lookup = {r["name"]: r for r in runs}
    return [run_lookup[name] for name in selected["run_name"].tolist()]

def _load_or_compute_traj_rmse(equation: str):
    """
    Try to load pre-computed trajectory RMSE from evaluation CSV.
    Falls back to computing on-the-fly if the file is missing or stale.
    """
    config_path = f"configs/training/{equation}_benchmark.yaml"
    cfg = load_config(config_path)
    experiment_name = cfg.get("experiment_name", "experiment")
    output_root = Path(cfg.get("output_dir", "outputs")) / experiment_name
    expected_param_index = TRAJ_PARAM_INDEX[equation]

    candidate_paths = [
        output_root / "evaluation" / "traj_rmse_per_trajectory.csv",
        output_root / "traj_rmse_per_trajectory.csv",
    ]
    candidate_paths.extend(sorted(output_root.rglob("traj_rmse_per_trajectory.csv")))

    seen_paths = []
    for traj_csv in candidate_paths:
        if traj_csv in seen_paths:
            continue
        seen_paths.append(traj_csv)
        if not traj_csv.exists():
            continue

        print(f"Loading pre-computed trajectory RMSE from {traj_csv}")
        traj_df = pd.read_csv(traj_csv)
        if "param_index" in traj_df.columns and (traj_df["param_index"] == expected_param_index).all():
            return traj_df
        print("Existing CSV does not match the requested parameter index; continuing search...")

    print(f"No matching cached trajectory RMSE found under {output_root}")
    print("Computing trajectory RMSE on-the-fly instead...")

    selected_runs = _build_selected_runs(equation)

    traj_df = ev.collect_traj_rmse_for_runs(
        run_cfgs=selected_runs,
        domains=("id", "od"),
        use_best=USE_BEST,
        param_key="parameter",
        param_index=expected_param_index,
        batch_size_test=500,
    )
    return traj_df

def _render_param_error_and_save(equation: str):
    traj_df = _load_or_compute_traj_rmse(equation)

    available_loaders = sorted(traj_df["loader"].unique().tolist())
    id_loader = "test_id"
    od_loader = "test_od" if "test_od" in available_loaders else next(
        (l for l in available_loaders if l != id_loader),
        None,
    )
    if od_loader is None:
        raise ValueError(f"No OD loader found. Available loaders: {available_loaders}")

    fig = vis.plot_param_vs_rmse_three_models_id_od(
        traj_df=traj_df,
        id_loader_name=id_loader,
        od_loader_name=od_loader,
        model_order=MODEL_ORDER,
        figsize=(10, 2.5),
        fontsize=9,
        param_key=PARAM_LABEL[equation],
    )

    output_dir = PROJECT_ROOT / "outputs/Figures"
    output_dir.mkdir(exist_ok=True)
    save_path = output_dir / SAVE_NAME[equation]
    fig.savefig(save_path, format="pdf", dpi=300, bbox_inches="tight", pad_inches=0.12)
    plt.show()
    print(f"Saved vector PDF to {save_path}")
    return fig, traj_df, save_path

1D advection

In [ ]:
fig_adv, traj_adv, save_adv = _render_param_error_and_save("advection")

1D Burgers

In [ ]:
fig_burg, traj_burg, save_burg = _render_param_error_and_save("burgers")

1D reaction-diffusion

In [ ]:
fig_rd1d, traj_rd1d, save_rd1d = _render_param_error_and_save("reactiondiffusion")

2D reaction-diffusion

In [ ]:
fig_rd2d, traj_rd2d, save_rd2d = _render_param_error_and_save("reactiondiffusion2d")

Merge into one stacked vector PDF with subtitles and gray separators

In [ ]:
import sys

try:
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation
except ImportError:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "pypdf"])
    from pypdf import PdfReader, PdfWriter, PageObject, Transformation

output_dir = PROJECT_ROOT / "outputs/Figures"
equation_order = ["advection", "burgers", "reactiondiffusion", "reactiondiffusion2d"]
pdf_paths = [output_dir / SAVE_NAME[e] for e in equation_order]

missing = [str(p) for p in pdf_paths if not p.exists()]
if missing:
    raise FileNotFoundError(f"Run Cells 5, 7, 9, and 11 first. Missing files: {missing}")

pages = [PdfReader(str(p)).pages[0] for p in pdf_paths]
widths = [float(p.mediabox.width) for p in pages]
heights = [float(p.mediabox.height) for p in pages]

max_width = max(widths)
gap = 18.0
title_band = 18.0
total_height = sum(heights) + gap * (len(pages) - 1) + title_band * len(pages)

merged_page = PageObject.create_blank_page(width=max_width, height=total_height)
cursor_y = total_height
section_tops = []
section_title_lines = []
section_bottoms = []

for page in pages:
    w = float(page.mediabox.width)
    h = float(page.mediabox.height)
    x = (max_width - w) / 2.0
    title_top = cursor_y
    title_bottom = title_top - title_band
    y = title_bottom - h

    merged_page.merge_transformed_page(
        page,
        Transformation().translate(tx=x, ty=y),
    )

    section_tops.append(title_top)
    section_title_lines.append((title_top + title_bottom) / 2.0)
    section_bottoms.append(y)
    cursor_y = y - gap

# Build a transparent vector overlay page for subtitles and gray separators.
overlay_fig = plt.figure(figsize=(max_width / 72.0, total_height / 72.0), dpi=72, facecolor="none")
overlay_fig.patch.set_alpha(0)
overlay_ax = overlay_fig.add_axes([0, 0, 1, 1])
overlay_ax.set_xlim(0, 1)
overlay_ax.set_ylim(0, 1)
overlay_ax.axis("off")
overlay_ax.set_facecolor("none")
overlay_ax.patch.set_alpha(0)

for i, eq in enumerate(equation_order):
    y_title_n = section_title_lines[i] / total_height
    overlay_ax.text(
        0.025,
        y_title_n,
        SUBTITLE[eq],
        ha="left",
        va="center",
        fontsize=12,
        color="black",
    )
    if i < len(equation_order) - 1:
        y_sep_n = (section_bottoms[i] - gap / 2.0) / total_height
        overlay_ax.plot([0.02, 0.98], [y_sep_n, y_sep_n], color="0.7", linewidth=1.0)

overlay_pdf = output_dir / "_paramerror_overlay_tmp.pdf"
overlay_fig.savefig(
    overlay_pdf,
    format="pdf",
    bbox_inches=None,
    pad_inches=0,
    transparent=True,
    facecolor="none",
    edgecolor="none",
)
plt.close(overlay_fig)

overlay_page = PdfReader(str(overlay_pdf)).pages[0]
merged_page.merge_transformed_page(overlay_page, Transformation())

writer = PdfWriter()
writer.add_page(merged_page)
merged_path = output_dir / "parametererror_stacked_merged.pdf"
with open(merged_path, "wb") as f:
    writer.write(f)

try:
    overlay_pdf.unlink()
except OSError:
    pass

print(f"Saved merged vector PDF to {merged_path}")